# TACO Dataset - Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimcev/DINOv3distill/blob/main/taco_eda.ipynb)

**TACO: Trash Annotations in Context** - Instance segmentation dataset of litter in the wild.

This notebook explores:
- Dataset structure and statistics
- Class distribution
- Image properties (sizes, aspect ratios)
- Annotation analysis (polygon masks)
- Sample visualizations

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q gdown matplotlib seaborn pandas opencv-python pyyaml

import os
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import cv2
from PIL import Image
import random

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("Setup complete!")

In [ ]:
# Download TACO dataset from Google Drive
import gdown
import zipfile

# Google Drive file ID
FILE_ID = "1HZBR53rs_igtr2ZlYqsLrS0uROCcTqzl"
ZIP_PATH = "taco.zip"
DATA_PATH = Path("./data/taco")

if not DATA_PATH.exists():
    print("Downloading TACO dataset from Google Drive...")
    gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", ZIP_PATH, quiet=False)
    
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('./data')
    print("Dataset ready!")
else:
    print(f"Dataset already exists at {DATA_PATH}")

!ls -la ./data/taco/

## 2. Dataset Overview

In [ ]:
# Load data.yaml
with open(DATA_PATH / "data.yaml", 'r') as f:
    data_config = yaml.safe_load(f)

# Extract class names
class_names = data_config['names']
num_classes = data_config['nc']

print(f"Number of classes: {num_classes}")
print(f"\nClass names:")
for i, name in enumerate(class_names):
    print(f"  {i:2d}: {name}")

In [ ]:
# Count images and labels in each split
splits = ['train', 'valid', 'test']
stats = {}

for split in splits:
    images_path = DATA_PATH / split / 'images'
    labels_path = DATA_PATH / split / 'labels'
    
    if images_path.exists():
        images = list(images_path.glob('*.jpg')) + list(images_path.glob('*.png')) + list(images_path.glob('*.JPG'))
        labels = list(labels_path.glob('*.txt')) if labels_path.exists() else []
        
        stats[split] = {
            'images': len(images),
            'labels': len(labels)
        }

# Display stats
stats_df = pd.DataFrame(stats).T
stats_df['total'] = stats_df['images']
print("Dataset Statistics:")
print(stats_df)
print(f"\nTotal images: {stats_df['images'].sum()}")

In [ ]:
# Visualize split distribution
fig, ax = plt.subplots(figsize=(8, 5))

splits_names = list(stats.keys())
counts = [stats[s]['images'] for s in splits_names]
colors = ['#2ecc71', '#3498db', '#e74c3c']

bars = ax.bar(splits_names, counts, color=colors, edgecolor='black', linewidth=1.2)

# Add value labels on bars
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, 
            str(count), ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_xlabel('Split', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('TACO Dataset - Split Distribution', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(counts) * 1.15)

plt.tight_layout()
plt.savefig('split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Class Distribution Analysis

In [ ]:
# Count annotations per class across all splits
class_counts = Counter()
annotations_per_image = []

for split in splits:
    labels_path = DATA_PATH / split / 'labels'
    if not labels_path.exists():
        continue
        
    for label_file in labels_path.glob('*.txt'):
        with open(label_file, 'r') as f:
            lines = f.readlines()
            annotations_per_image.append(len(lines))
            
            for line in lines:
                parts = line.strip().split()
                if parts:
                    class_id = int(parts[0])
                    class_counts[class_id] += 1

print(f"Total annotations: {sum(class_counts.values())}")
print(f"Average annotations per image: {np.mean(annotations_per_image):.2f}")
print(f"Max annotations in single image: {max(annotations_per_image)}")
print(f"Min annotations in single image: {min(annotations_per_image)}")

In [ ]:
# Create class distribution dataframe
class_data = []
for class_id, count in sorted(class_counts.items()):
    class_data.append({
        'class_id': class_id,
        'class_name': class_names[class_id] if class_id < len(class_names) else f'Unknown_{class_id}',
        'count': count
    })

class_df = pd.DataFrame(class_data)
class_df = class_df.sort_values('count', ascending=False).reset_index(drop=True)

print("Top 15 classes by annotation count:")
print(class_df.head(15).to_string(index=False))

print("\nBottom 10 classes by annotation count:")
print(class_df.tail(10).to_string(index=False))

In [ ]:
# Visualize class distribution (top 20)
fig, ax = plt.subplots(figsize=(14, 8))

top_n = 20
top_classes = class_df.head(top_n)

bars = ax.barh(range(len(top_classes)), top_classes['count'], color=plt.cm.viridis(np.linspace(0.2, 0.8, top_n)))
ax.set_yticks(range(len(top_classes)))
ax.set_yticklabels(top_classes['class_name'])
ax.invert_yaxis()

# Add value labels
for i, (bar, count) in enumerate(zip(bars, top_classes['count'])):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, 
            str(count), va='center', fontsize=10)

ax.set_xlabel('Number of Annotations', fontsize=12)
ax.set_title(f'TACO Dataset - Top {top_n} Classes by Annotation Count', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Class imbalance analysis
print("Class Imbalance Analysis:")
print(f"  Most common class: {class_df.iloc[0]['class_name']} ({class_df.iloc[0]['count']} annotations)")
print(f"  Least common class: {class_df.iloc[-1]['class_name']} ({class_df.iloc[-1]['count']} annotations)")
print(f"  Imbalance ratio: {class_df.iloc[0]['count'] / class_df.iloc[-1]['count']:.1f}x")

# Classes with fewer than 10 annotations
rare_classes = class_df[class_df['count'] < 10]
print(f"\nClasses with < 10 annotations: {len(rare_classes)}")
if len(rare_classes) > 0:
    print(rare_classes[['class_name', 'count']].to_string(index=False))

## 4. Image Properties Analysis

In [ ]:
# Analyze image dimensions
image_sizes = []
aspect_ratios = []

# Sample images from train split
train_images = list((DATA_PATH / 'train' / 'images').glob('*'))
sample_size = min(500, len(train_images))  # Sample up to 500 images
sampled_images = random.sample(train_images, sample_size)

for img_path in sampled_images:
    try:
        with Image.open(img_path) as img:
            w, h = img.size
            image_sizes.append((w, h))
            aspect_ratios.append(w / h)
    except:
        continue

widths = [s[0] for s in image_sizes]
heights = [s[1] for s in image_sizes]

print(f"Analyzed {len(image_sizes)} images")
print(f"\nWidth statistics:")
print(f"  Min: {min(widths)}, Max: {max(widths)}, Mean: {np.mean(widths):.0f}, Std: {np.std(widths):.0f}")
print(f"\nHeight statistics:")
print(f"  Min: {min(heights)}, Max: {max(heights)}, Mean: {np.mean(heights):.0f}, Std: {np.std(heights):.0f}")
print(f"\nAspect ratio statistics:")
print(f"  Min: {min(aspect_ratios):.2f}, Max: {max(aspect_ratios):.2f}, Mean: {np.mean(aspect_ratios):.2f}")

In [ ]:
# Visualize image dimensions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Width distribution
axes[0].hist(widths, bins=30, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(widths), color='red', linestyle='--', label=f'Mean: {np.mean(widths):.0f}')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Count')
axes[0].set_title('Image Width Distribution')
axes[0].legend()

# Height distribution
axes[1].hist(heights, bins=30, color='#2ecc71', edgecolor='black', alpha=0.7)
axes[1].axvline(np.mean(heights), color='red', linestyle='--', label=f'Mean: {np.mean(heights):.0f}')
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Count')
axes[1].set_title('Image Height Distribution')
axes[1].legend()

# Aspect ratio distribution
axes[2].hist(aspect_ratios, bins=30, color='#e74c3c', edgecolor='black', alpha=0.7)
axes[2].axvline(1.0, color='blue', linestyle='--', label='Square (1.0)')
axes[2].axvline(np.mean(aspect_ratios), color='red', linestyle='--', label=f'Mean: {np.mean(aspect_ratios):.2f}')
axes[2].set_xlabel('Aspect Ratio (W/H)')
axes[2].set_ylabel('Count')
axes[2].set_title('Aspect Ratio Distribution')
axes[2].legend()

plt.tight_layout()
plt.savefig('image_dimensions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter plot of image dimensions
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(widths, heights, alpha=0.5, c='#3498db', edgecolors='none', s=30)
ax.set_xlabel('Width (pixels)', fontsize=12)
ax.set_ylabel('Height (pixels)', fontsize=12)
ax.set_title('Image Dimensions Scatter Plot', fontsize=14, fontweight='bold')

# Add common aspect ratio lines
max_dim = max(max(widths), max(heights))
ax.plot([0, max_dim], [0, max_dim], 'r--', alpha=0.5, label='1:1')
ax.plot([0, max_dim], [0, max_dim * 3/4], 'g--', alpha=0.5, label='4:3')
ax.plot([0, max_dim], [0, max_dim * 9/16], 'b--', alpha=0.5, label='16:9')
ax.legend()

plt.tight_layout()
plt.savefig('image_dimensions_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Annotation Analysis (Polygon Masks)

In [ ]:
# Analyze polygon complexity (number of vertices)
polygon_vertices = []
polygon_areas = []  # Approximate areas

train_labels = list((DATA_PATH / 'train' / 'labels').glob('*.txt'))
sample_labels = random.sample(train_labels, min(200, len(train_labels)))

for label_file in sample_labels:
    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) > 5:  # Has polygon coordinates
                # Number of vertices = (total parts - 1) / 2
                num_vertices = (len(parts) - 1) // 2
                polygon_vertices.append(num_vertices)
                
                # Extract coordinates and calculate approximate area
                coords = [float(x) for x in parts[1:]]
                xs = coords[0::2]
                ys = coords[1::2]
                
                # Shoelace formula for polygon area
                n = len(xs)
                area = 0.5 * abs(sum(xs[i]*ys[(i+1)%n] - xs[(i+1)%n]*ys[i] for i in range(n)))
                polygon_areas.append(area)

print(f"Analyzed {len(polygon_vertices)} polygon annotations")
print(f"\nPolygon vertices statistics:")
print(f"  Min: {min(polygon_vertices)}, Max: {max(polygon_vertices)}")
print(f"  Mean: {np.mean(polygon_vertices):.1f}, Median: {np.median(polygon_vertices):.0f}")
print(f"\nPolygon area statistics (normalized):")
print(f"  Min: {min(polygon_areas):.4f}, Max: {max(polygon_areas):.4f}")
print(f"  Mean: {np.mean(polygon_areas):.4f}, Median: {np.median(polygon_areas):.4f}")

In [ ]:
# Visualize polygon statistics
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Vertices distribution
axes[0].hist(polygon_vertices, bins=30, color='#9b59b6', edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(polygon_vertices), color='red', linestyle='--', label=f'Mean: {np.mean(polygon_vertices):.1f}')
axes[0].set_xlabel('Number of Vertices')
axes[0].set_ylabel('Count')
axes[0].set_title('Polygon Complexity (Vertices per Mask)')
axes[0].legend()

# Area distribution
axes[1].hist(polygon_areas, bins=30, color='#f39c12', edgecolor='black', alpha=0.7)
axes[1].axvline(np.mean(polygon_areas), color='red', linestyle='--', label=f'Mean: {np.mean(polygon_areas):.4f}')
axes[1].set_xlabel('Normalized Area')
axes[1].set_ylabel('Count')
axes[1].set_title('Polygon Area Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('polygon_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Annotations per image distribution
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(annotations_per_image, bins=range(0, max(annotations_per_image)+2), 
        color='#1abc9c', edgecolor='black', alpha=0.7, align='left')
ax.axvline(np.mean(annotations_per_image), color='red', linestyle='--', 
           label=f'Mean: {np.mean(annotations_per_image):.1f}')
ax.set_xlabel('Number of Annotations per Image', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Annotations per Image', fontsize=14, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('annotations_per_image.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Sample Visualizations

In [ ]:
def draw_polygon_mask(image, label_path, class_names, alpha=0.4):
    """Draw polygon masks on image."""
    img = image.copy()
    h, w = img.shape[:2]
    
    # Generate colors for each class
    np.random.seed(42)
    colors = {i: tuple(np.random.randint(0, 255, 3).tolist()) for i in range(len(class_names))}
    
    if not os.path.exists(label_path):
        return img
    
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
                
            class_id = int(parts[0])
            coords = [float(x) for x in parts[1:]]
            
            # Convert normalized coordinates to pixel coordinates
            points = []
            for i in range(0, len(coords), 2):
                x = int(coords[i] * w)
                y = int(coords[i+1] * h)
                points.append([x, y])
            
            points = np.array(points, dtype=np.int32)
            
            # Draw filled polygon with transparency
            overlay = img.copy()
            color = colors.get(class_id, (255, 255, 255))
            cv2.fillPoly(overlay, [points], color)
            img = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)
            
            # Draw polygon outline
            cv2.polylines(img, [points], True, color, 2)
            
            # Add class label
            label = class_names[class_id] if class_id < len(class_names) else f'Class {class_id}'
            centroid = points.mean(axis=0).astype(int)
            cv2.putText(img, label[:15], tuple(centroid), cv2.FONT_HERSHEY_SIMPLEX, 
                       0.4, (255, 255, 255), 1, cv2.LINE_AA)
    
    return img

In [ ]:
# Visualize random samples with masks
train_images = list((DATA_PATH / 'train' / 'images').glob('*'))
sample_images = random.sample(train_images, min(8, len(train_images)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_images):
    # Load image
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Get label path
    label_path = str(img_path).replace('/images/', '/labels/').replace('.jpg', '.txt').replace('.JPG', '.txt').replace('.png', '.txt')
    
    # Draw masks
    img_with_masks = draw_polygon_mask(img, label_path, class_names)
    
    ax.imshow(img_with_masks)
    ax.axis('off')
    ax.set_title(img_path.name[:30] + '...', fontsize=10)

plt.suptitle('TACO Dataset - Sample Images with Segmentation Masks', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_visualizations.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary Statistics

In [ ]:
# Print summary
print("="*60)
print("TACO DATASET - SUMMARY")
print("="*60)
print(f"\nDataset: TACO (Trash Annotations in Context)")
print(f"Task: Instance Segmentation")
print(f"License: CC BY 4.0")
print(f"\n--- Images ---")
print(f"Total images: {sum(stats[s]['images'] for s in splits)}")
print(f"  Train: {stats['train']['images']}")
print(f"  Valid: {stats['valid']['images']}")
print(f"  Test: {stats['test']['images']}")
print(f"\n--- Classes ---")
print(f"Number of classes: {num_classes}")
print(f"Total annotations: {sum(class_counts.values())}")
print(f"Most common: {class_df.iloc[0]['class_name']} ({class_df.iloc[0]['count']})")
print(f"Least common: {class_df.iloc[-1]['class_name']} ({class_df.iloc[-1]['count']})")
print(f"\n--- Annotations ---")
print(f"Avg annotations/image: {np.mean(annotations_per_image):.1f}")
print(f"Avg polygon vertices: {np.mean(polygon_vertices):.1f}")
print(f"\n--- Image Properties ---")
print(f"Avg dimensions: {np.mean(widths):.0f} x {np.mean(heights):.0f}")
print(f"Avg aspect ratio: {np.mean(aspect_ratios):.2f}")
print("="*60)